In [48]:
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from toolbox import (
    Masking,
    combined_frequency_roc_analysis,
    compute_ROC_patient_split,
    find_all_indices,
    get_freqs,
    hot_coding,
)
from toolbox.loading import load_data
from toolbox.visualization import color_palette_full

In [49]:
# load masking
IIVV = np.loadtxt("source/IIVV_patterns.txt", dtype=int)
mask = Masking(IIVV)

In [50]:
(
    py_idx,
    patient_no,
    patient_sample,
    Zdata,
    Avg_Cutpats,
    LesionAssessment,
    labels_healthy_vs_cancer,
    labels_cancer_vs_hbd,
    labels_MC,
) = load_data(path="data", Cutpats_threshold=60, norm="freq")


	Info:	np.abs(Zdata)

Remaining data samples: 244 
	Info:	Select only data Avg_Cutpats <60%

Remaining data samples: 229 

	Info:	z-score Zdata
before: 434.20845093924504 699.3585400487347
Normalized along the frequency axis.

after: -4.3347838749237164e-17 1.000000000000001

	Info:	np.nan_to_num(Zdata)

Remaining data samples: 229 


**Select classification basis task `class_task`**

- `"healthy_vs_cancer"`
- `"cancer_vs_hbd"`
- `"MC"`

In [51]:
class_task = "healthy_vs_cancer"
# class_task = "cancer_vs_hbd"
# class_task = "MC"

In [52]:
if class_task == "healthy_vs_cancer":
    print("Healthy vs. cancer\n")
    select_idx = find_all_indices(
        labels_healthy_vs_cancer, ["Healthy", "Cancer"], split=False
    )
    labels = labels_healthy_vs_cancer[select_idx]


elif class_task == "cancer_vs_hbd":
    print("\nHealthy vs. HDB\n")
    select_idx = find_all_indices(
        labels_cancer_vs_hbd, ["Non-cancer", "Cancer"], split=False
    )
    labels = labels_cancer_vs_hbd[select_idx]

elif class_task == "MC":
    print("\nMC\n")
    select_idx = find_all_indices(
        labels_MC,
        ["Healthy", "Cancer", "Mild dysplasia", "Severe dysplasia", "Benign"],
        split=False,
    )
    labels = labels_MC[select_idx]

Case = patient_no[select_idx]
X = Zdata[select_idx, ...]
Y, inverse_mapping = hot_coding(labels)
print("\nInverse mapping:", inverse_mapping)
print(X.shape, Y.shape, Case.shape)

Healthy vs. cancer

Healthy: 89 entries
Cancer: 91 entries
Available cases: ['Cancer' 'Healthy']
Coding: -> forward
{np.str_('Cancer'): [1, 0], np.str_('Healthy'): [0, 1]}

Inverse mapping: {(1, 0): 'Cancer', (0, 1): 'Healthy'}
(180, 31, 7728) (180, 2) (180,)


In [53]:
# load frequency selection
tmp = np.load("results/freq_fold_results.npz", allow_pickle=True)
top_freqs_baseline = tmp["top_freqs_baseline"]
top_2 = top_freqs_baseline[:2]
top_3 = top_freqs_baseline[:3]
top_5 = top_freqs_baseline[:5]
top_12 = top_freqs_baseline[:12]

Freq_sel = {
    "Top 2": top_2,
    "Top 3": top_3,
    "Top 5": top_5,
    "Top 12": top_12,
    "All": top_freqs_baseline,  # np.arange(31), # it doesn't matter
}

## Classification model approaches
- SVM (like in freq. selection)
- Random forest
- CNN network

In general TP rate

In [54]:
Freq_sel

{'Top 2': array([30,  0]),
 'Top 3': array([30,  0, 27]),
 'Top 5': array([30,  0, 27, 12, 29]),
 'Top 12': array([30,  0, 27, 12, 29, 28, 13, 16, 20,  8, 26, 23]),
 'All': array([30,  0, 27, 12, 29, 28, 13, 16, 20,  8, 26, 23, 11, 24, 25, 15, 10,
        17, 22,  1, 21, 19, 14,  2,  7, 18,  3,  4,  5,  9,  6])}

In [14]:
random_state = 0
n_folds = 5

In [39]:
unique_patients = np.unique(Case)
np.random.seed(random_state)

# Check if Y is in one-hot encoding
if len(Y.shape) == 1 or Y.shape[1] == 1:
    raise ValueError("Labels (Y) must be in one-hot encoding format")

n_classes = Y.shape[1]

# Initialize arrays to store fold-specific rankings and overall frequency scores
all_fold_rankings = []

# Store AUC results for each fold
fold_aucs = {i: [] for i in range(n_folds)}
final_aucs = np.zeros(n_folds)

# Custom splitting to ensure 70/30 ratio
patient_indices = np.arange(len(unique_patients))
np.random.shuffle(patient_indices)

# Split these indices into n_folds
fold_size = len(patient_indices) // n_folds
folds = []

for i in range(n_folds):
    if i == n_folds - 1:
        # Last fold gets all remaining patients
        test_indices = patient_indices[i * fold_size :]
    else:
        test_indices = patient_indices[i * fold_size : (i + 1) * fold_size]
    # Training indices are all other indices
    train_indices = np.setdiff1d(patient_indices, test_indices)
    folds.append((train_indices, test_indices))

In [40]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_curve, roc_auc_score, auc
from sklearn import svm

In [43]:
# Loop through each fold
fold_auc_results = []

for fold_idx, (train_patients_idx, test_patients_idx) in enumerate(folds):
    fold = fold_idx + 1

    # Get patient IDs for training and testing
    train_patients = unique_patients[train_patients_idx]
    test_patients = unique_patients[test_patients_idx]
    print("Train patients:", train_patients)
    print("Test patients:", test_patients)

    # Create masks for training and testing samples
    train_mask = np.isin(Case, train_patients)
    test_mask = np.isin(Case, test_patients)

    # Verify split is close to 70/30
    train_pct = train_mask.mean() * 100
    test_pct = test_mask.mean() * 100
    print(f"Fold {fold}: {train_pct:.1f}% train / {test_pct:.1f}% test")

    # Count samples and patients in each set
    train_samples = np.sum(train_mask)
    test_samples = np.sum(test_mask)
    train_patient_count = len(train_patients)
    test_patient_count = len(test_patients)

    print(f"Fold {fold}: {train_pct:.1f}% train / {test_pct:.1f}% test")
    print(f"  Training: {train_samples} samples from {train_patient_count} patients")
    print(f"  Testing:  {test_samples} samples from {test_patient_count} patients")

    # Get training and testing data
    X_train = X[train_mask]
    X_test = X[test_mask]
    Y_train = Y[train_mask]
    Y_test = Y[test_mask]

    # here the frequency and IIVV selection is done
    X_train_filtered = X_train[:, :, :]
    X_test_filtered = X_test[:, :, :]

    # Step 1: Reshape and clean training data for PCA
    X_train_reshaped = X_train.transpose(0, 2, 1).reshape(
        -1, X_train.shape[1]
    )  # (train_samples * patterns, freqs)
    X_train_clean = (
        X_train_reshaped  # remove_nan_rows_for_pca(X_train_reshaped, verbose=False)
    )

    # Reshape for classifier
    X_train_flat = X_train_filtered.reshape(X_train_filtered.shape[0], -1)
    X_test_flat = X_test_filtered.reshape(X_test_filtered.shape[0], -1)

    # Replace NaN values with 0
    X_train_flat = np.nan_to_num(X_train_flat)
    X_test_flat = np.nan_to_num(X_test_flat)

    # Train classifier
    classifier = OneVsRestClassifier(
        svm.SVC(kernel="linear", probability=True, random_state=random_state)
    )
    # Fit and predict
    y_score = classifier.fit(X_train_flat, Y_train).predict_proba(X_test_flat)

    # Compute ROC AUC for each class
    roc_auc_scores = []
    for i in range(n_classes):
        score = roc_auc_score(Y_test[:, i], y_score[:, i])
        roc_auc_scores.append(score)

    # Calculate average AUC across classes
    avg_auc = np.mean(roc_auc_scores) if roc_auc_scores else 0
    fold_auc_results.append(avg_auc)

    # Store this fold's AUC results
    fold_aucs[fold_idx] = fold_auc_results

Train patients: [ 0  1  3  4  6  7 10 11 12 13 15 16 19 20 21 22 23 25 27 28 29 30 31 32
 34 35 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 55 56 57 58 60
 61 62 66 67 68 69 70 72 73 74 75 76 77 78 79 81 84 85 86 87 88 89 90 91
 92 95 96 97]
Test patients: [ 2 33 59 18 14 64 65 83 36 82 80  8 54 94 99 63  9 24]
Fold 1: 77.2% train / 22.8% test
Fold 1: 77.2% train / 22.8% test
  Training: 139 samples from 76 patients
  Testing:  41 samples from 18 patients
Train patients: [ 0  1  2  4  6  8  9 10 11 12 13 14 15 16 18 19 21 22 23 24 25 28 31 32
 33 34 35 36 37 38 39 40 41 42 43 44 47 49 50 52 53 54 55 56 57 58 59 60
 61 62 63 64 65 66 67 68 70 72 73 74 75 76 79 80 82 83 84 85 87 89 92 94
 95 96 97 99]
Test patients: [77 48 29 46 27  7 45 51 78 86 81 91 90  3 69 88 30 20]
Fold 2: 81.1% train / 18.9% test
Fold 2: 81.1% train / 18.9% test
  Training: 146 samples from 76 patients
  Testing:  34 samples from 18 patients
Train patients: [ 1  2  3  7  8  9 10 11 12 13 14 15 18 20 21 22 2

In [ ]:
# Get unique patient IDs